In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import json
import math
import time
import getpass
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

import psycopg2
from psycopg2 import sql

from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

PROJECT_ROOT = Path(r"C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
PREDICTIONS_DIR = DATA_DIR / "predictions"

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

for directory in [
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    PREDICTIONS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "ecommerce_ai_db",
    "user": input("PostgreSQL Username: ").strip(),
    "password": getpass.getpass("PostgreSQL Password: ")
}

CURRENT_TIMESTAMP = datetime.now()

print("=" * 80)
print("ECOMMERCE-AI-PLATFORM")
print("Notebook 14 - AI Business Insights Engine")
print("=" * 80)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Database     : {DB_CONFIG['database']}")
print(f"Host         : {DB_CONFIG['host']}")
print(f"Port         : {DB_CONFIG['port']}")
print(f"Started      : {CURRENT_TIMESTAMP}")

print("\nEnvironment Ready.")

PostgreSQL Username:  postgres
PostgreSQL Password:  ········


ECOMMERCE-AI-PLATFORM
Notebook 14 - AI Business Insights Engine
Project Root : C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM
Database     : ecommerce_ai_db
Host         : localhost
Port         : 5432
Started      : 2026-07-31 18:01:53.574094

Environment Ready.


In [2]:
def create_psycopg2_connection(config: dict):

    connection = psycopg2.connect(
        host=config["host"],
        port=config["port"],
        dbname=config["database"],
        user=config["user"],
        password=config["password"]
    )

    connection.autocommit = False

    return connection


def create_sqlalchemy_engine(config: dict) -> Engine:

    connection_url = (
        f"postgresql+psycopg2://"
        f"{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}"
        f"/{config['database']}"
    )

    engine = create_engine(
        connection_url,
        future=True,
        pool_pre_ping=True
    )

    return engine


try:

    conn = create_psycopg2_connection(DB_CONFIG)
    engine = create_sqlalchemy_engine(DB_CONFIG)

    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))

    print("Database connection successful.")

except Exception as e:

    raise RuntimeError(
        f"Unable to establish PostgreSQL connection.\n{e}"
    )

Database connection successful.


In [3]:
def rollback_transaction(connection):

    try:
        connection.rollback()
    except Exception:
        pass


def execute_query(connection, query, params=None):

    try:

        with connection.cursor() as cursor:

            cursor.execute(query, params)

            if cursor.description is None:
                connection.commit()
                return None

            columns = [column[0] for column in cursor.description]

            rows = cursor.fetchall()

            connection.commit()

            return pd.DataFrame(rows, columns=columns)

    except Exception as error:

        rollback_transaction(connection)

        raise RuntimeError(error)


def dataframe_to_postgresql(
    dataframe: pd.DataFrame,
    table_name: str,
    schema_name: str,
    engine: Engine,
    if_exists: str = "replace"
):

    dataframe.to_sql(
        name=table_name,
        schema=schema_name,
        con=engine,
        index=False,
        if_exists=if_exists,
        method="multi",
        chunksize=5000
    )

In [4]:
DATABASE_DISCOVERY_SQL = """
SELECT
    n.nspname AS schema_name,
    c.relname AS object_name,
    CASE
        WHEN c.relkind='r' THEN 'table'
        WHEN c.relkind='v' THEN 'view'
        WHEN c.relkind='m' THEN 'materialized_view'
        ELSE c.relkind::text
    END AS object_type
FROM pg_class c
JOIN pg_namespace n
ON n.oid=c.relnamespace
WHERE
n.nspname NOT IN
(
'pg_catalog',
'information_schema'
)
ORDER BY
schema_name,
object_type,
object_name;
"""

database_objects = execute_query(
    conn,
    DATABASE_DISCOVERY_SQL
)

if database_objects.empty:
    raise RuntimeError("No accessible database objects were discovered.")

display(database_objects.head(100))

print(f"\nDiscovered Objects : {len(database_objects)}")
print(f"Schemas            : {database_objects['schema_name'].nunique()}")

,schema_name,object_name,object_type
0,analytics,clv_overall_summary,table
1,analytics,clv_segment_summary,table
2,analytics,clv_validation,table
3,analytics,customer_churn_model_metrics,table
4,analytics,customer_churn_predictions,table
5,analytics,customer_lifetime_value,table
6,analytics,customer_segmentation,table
7,analytics,customer_segmentation_model_evaluation,table
8,analytics,segment_distribution,table
9,analytics,segment_feature_means,table



Discovered Objects : 193
Schemas            : 7


In [5]:
COLUMN_DISCOVERY_SQL = """
SELECT

table_schema,

table_name,

column_name,

ordinal_position,

data_type,

is_nullable

FROM information_schema.columns

WHERE

table_schema NOT IN

(

'pg_catalog',

'information_schema'

)

ORDER BY

table_schema,

table_name,

ordinal_position;
"""

database_columns = execute_query(
    conn,
    COLUMN_DISCOVERY_SQL
)

if database_columns.empty:
    raise RuntimeError("Column discovery failed.")

display(database_columns.head(100))

print(f"Columns discovered : {len(database_columns):,}")

,table_schema,table_name,column_name,ordinal_position,data_type,is_nullable
0,analytics,clv_overall_summary,metric,1,text,YES
1,analytics,clv_overall_summary,value,2,text,YES
2,analytics,clv_segment_summary,clv_segment,1,text,YES
3,analytics,clv_segment_summary,customer_count,2,bigint,YES
4,analytics,clv_segment_summary,total_historical_clv,3,double precision,YES
5,analytics,clv_segment_summary,average_historical_clv,4,double precision,YES
6,analytics,clv_segment_summary,median_historical_clv,5,double precision,YES
7,analytics,clv_segment_summary,minimum_historical_clv,6,double precision,YES
8,analytics,clv_segment_summary,maximum_historical_clv,7,double precision,YES
9,analytics,clv_segment_summary,clv_value_share,8,double precision,YES


Columns discovered : 378


In [6]:
PRIMARY_KEY_SQL = """
SELECT

tc.table_schema,
tc.table_name,
kcu.column_name,
kcu.ordinal_position,
tc.constraint_name

FROM information_schema.table_constraints tc

JOIN information_schema.key_column_usage kcu

ON tc.constraint_name = kcu.constraint_name
AND tc.table_schema = kcu.table_schema

WHERE tc.constraint_type='PRIMARY KEY'

ORDER BY

tc.table_schema,
tc.table_name,
kcu.ordinal_position;
"""

primary_keys = execute_query(
    conn,
    PRIMARY_KEY_SQL
)

display(primary_keys)

print(f"Primary Key Columns : {len(primary_keys):,}")

,table_schema,table_name,column_name,ordinal_position,constraint_name
0,pg_catalog,pg_aggregate,aggfnoid,1,pg_aggregate_fnoid_index
1,pg_catalog,pg_am,oid,1,pg_am_oid_index
2,pg_catalog,pg_amop,oid,1,pg_amop_oid_index
3,pg_catalog,pg_amproc,oid,1,pg_amproc_oid_index
4,pg_catalog,pg_attrdef,oid,1,pg_attrdef_oid_index
5,pg_catalog,pg_attribute,attrelid,1,pg_attribute_relid_attnum_index
6,pg_catalog,pg_attribute,attnum,2,pg_attribute_relid_attnum_index
7,pg_catalog,pg_auth_members,oid,1,pg_auth_members_oid_index
8,pg_catalog,pg_authid,oid,1,pg_authid_oid_index
9,pg_catalog,pg_cast,oid,1,pg_cast_oid_index


Primary Key Columns : 82


In [7]:
FOREIGN_KEY_SQL = """
SELECT

tc.table_schema,

tc.table_name,

kcu.column_name,

ccu.table_schema AS referenced_schema,

ccu.table_name AS referenced_table,

ccu.column_name AS referenced_column,

tc.constraint_name

FROM information_schema.table_constraints tc

JOIN information_schema.key_column_usage kcu

ON tc.constraint_name=kcu.constraint_name

AND tc.table_schema=kcu.table_schema

JOIN information_schema.constraint_column_usage ccu

ON ccu.constraint_name=tc.constraint_name

AND ccu.table_schema=tc.table_schema

WHERE tc.constraint_type='FOREIGN KEY'

ORDER BY

tc.table_schema,

tc.table_name;
"""

foreign_keys = execute_query(
    conn,
    FOREIGN_KEY_SQL
)

display(foreign_keys)

print(f"Foreign Keys : {len(foreign_keys):,}")

,table_schema,table_name,column_name,referenced_schema,referenced_table,referenced_column,constraint_name


Foreign Keys : 0


In [8]:
row_statistics = []

for _, row in database_objects.iterrows():

    if row["object_type"] != "table":
        continue

    schema_name = row["schema_name"]
    table_name = row["object_name"]

    query = sql.SQL("""
        SELECT COUNT(*) AS row_count
        FROM {}.{}
    """).format(
        sql.Identifier(schema_name),
        sql.Identifier(table_name)
    )

    try:

        result = execute_query(
            conn,
            query.as_string(conn)
        )

        row_statistics.append({
            "schema_name": schema_name,
            "table_name": table_name,
            "row_count": int(result.iloc[0, 0])
        })

    except Exception:

        row_statistics.append({
            "schema_name": schema_name,
            "table_name": table_name,
            "row_count": np.nan
        })

row_statistics = pd.DataFrame(row_statistics)

display(row_statistics)

print(f"Validated Tables : {len(row_statistics)}")

,schema_name,table_name,row_count
0,analytics,clv_overall_summary,9
1,analytics,clv_segment_summary,3
2,analytics,clv_validation,9
3,analytics,customer_churn_model_metrics,1
4,analytics,customer_churn_predictions,86924
5,analytics,customer_lifetime_value,99441
6,analytics,customer_segmentation,99441
7,analytics,customer_segmentation_model_evaluation,7
8,analytics,segment_distribution,2
9,analytics,segment_feature_means,2


Validated Tables : 39


In [9]:
table_validation = []

for _, table in row_statistics.iterrows():

    schema_name = table["schema_name"]
    table_name = table["table_name"]

    columns = database_columns[
        (database_columns.table_schema == schema_name)
        &
        (database_columns.table_name == table_name)
    ]

    duplicate_columns = columns.column_name.duplicated().sum()

    table_validation.append({

        "schema_name": schema_name,

        "table_name": table_name,

        "row_count": table["row_count"],

        "column_count": len(columns),

        "duplicate_column_names": duplicate_columns,

        "nullable_columns":

        int(
            (
                columns["is_nullable"] == "YES"
            ).sum()
        )

    })

table_validation = pd.DataFrame(table_validation)

display(table_validation)

,schema_name,table_name,row_count,column_count,duplicate_column_names,nullable_columns
0,analytics,clv_overall_summary,9,2,0,2
1,analytics,clv_segment_summary,3,9,0,9
2,analytics,clv_validation,9,2,0,2
3,analytics,customer_churn_model_metrics,1,10,0,10
4,analytics,customer_churn_predictions,86924,9,0,9
5,analytics,customer_lifetime_value,99441,6,0,6
6,analytics,customer_segmentation,99441,18,0,18
7,analytics,customer_segmentation_model_evaluation,7,7,0,7
8,analytics,segment_distribution,2,4,0,4
9,analytics,segment_feature_means,2,13,0,13


In [10]:
NOTEBOOK_OUTPUT_PATTERNS = {

    "sales_analytics": [

        "sales",
        "analytics",
        "revenue",
        "monthly_sales"

    ],

    "customer_segmentation": [

        "segment",
        "cluster"

    ],

    "customer_lifetime_value": [

        "clv",
        "lifetime"

    ],

    "customer_churn": [

        "churn"

    ],

    "recommendation": [

        "recommend",

        "recommendation"

    ],

    "forecast": [

        "forecast",

        "prediction",

        "future"

    ],

    "feature_engineering": [

        "feature"

    ]

}

validated_outputs = {}

for notebook_name, keywords in NOTEBOOK_OUTPUT_PATTERNS.items():

    discovered = []

    for _, obj in database_objects.iterrows():

        object_name = obj["object_name"].lower()

        if any(keyword in object_name for keyword in keywords):

            discovered.append({

                "schema": obj["schema_name"],

                "table": obj["object_name"],

                "type": obj["object_type"]

            })

    validated_outputs[notebook_name] = pd.DataFrame(discovered)

for notebook_name, dataframe in validated_outputs.items():

    print("\n" + "=" * 70)

    print(notebook_name.upper())

    print("=" * 70)

    if dataframe.empty:

        print("No validated outputs discovered.")

    else:

        display(dataframe)


SALES_ANALYTICS


,schema,table,type
0,feature_engineered,monthly_sales_features,table
1,forecasting,future_sales_forecast,table
2,forecasting,historical_sales_forecast,table
3,sales_analytics,monthly_sales,table
4,sales_analytics,overall_sales_kpis,table



CUSTOMER_SEGMENTATION


,schema,table,type
0,analytics,clv_segment_summary,table
1,analytics,customer_segmentation,table
2,analytics,customer_segmentation_model_evaluation,table
3,analytics,segment_distribution,table
4,analytics,segment_feature_means,table
5,analytics,segment_summary,table



CUSTOMER_LIFETIME_VALUE


...

,schema,table,type
0,recommendations,powerbi_product_recommendations,table
1,recommendations,product_recommendations,table
2,recommendations,recommendation_data_quality,table
3,recommendations,recommendation_evaluation,table



FORECAST


,schema,table,type
0,analytics,customer_churn_predictions,table
1,forecasting,future_sales_forecast,table
2,forecasting,historical_sales_forecast,table
3,public,customer_churn_predictions,table



FEATURE_ENGINEERING


,schema,table,type
0,analytics,segment_feature_means,table
1,feature_engineered,category_features,table
2,feature_engineered,customer_features,table
3,feature_engineered,monthly_sales_features,table
4,feature_engineered,order_features,table
5,feature_engineered,product_features,table
6,feature_engineered,seller_features,table


In [11]:
validated_tables = {}

for notebook_name, discovery_df in validated_outputs.items():

    notebook_tables = {}

    if discovery_df.empty:

        validated_tables[notebook_name] = notebook_tables

        continue

    for _, table in discovery_df.iterrows():

        schema_name = table["schema"]
        table_name = table["table"]

        query = sql.SQL("""

            SELECT *

            FROM {}.{}

            LIMIT 50000

        """).format(

            sql.Identifier(schema_name),

            sql.Identifier(table_name)

        )

        try:

            dataframe = execute_query(
                conn,
                query.as_string(conn)
            )

            notebook_tables[f"{schema_name}.{table_name}"] = dataframe

        except Exception:

            continue

    validated_tables[notebook_name] = notebook_tables

print("Validated notebook outputs loaded successfully.")

Validated notebook outputs loaded successfully.


In [12]:
data_quality_summary = []

for notebook_name, tables in validated_tables.items():

    for table_name, dataframe in tables.items():

        quality = {

            "notebook": notebook_name,

            "table": table_name,

            "rows": len(dataframe),

            "columns": len(dataframe.columns),

            "duplicate_rows": int(dataframe.duplicated().sum()),

            "missing_cells": int(dataframe.isna().sum().sum()),

            "missing_percentage":

            round(

                dataframe.isna().sum().sum()

                /

                max(dataframe.size, 1)

                *

                100,

                2

            )

        }

        data_quality_summary.append(quality)

data_quality_summary = pd.DataFrame(data_quality_summary)

display(data_quality_summary.sort_values(
    "missing_percentage",
    ascending=False
))

,notebook,table,rows,columns,duplicate_rows,missing_cells,missing_percentage
22,recommendation,recommendations.recommendation_evaluation,1,6,0,4,66.67
14,customer_lifetime_value,analytics.customer_lifetime_value,50000,6,0,50000,16.67
0,sales_analytics,feature_engineered.monthly_sales_features,25,11,0,0,0.00
25,forecast,forecasting.historical_sales_forecast,1299,6,0,0,0.00
20,recommendation,recommendations.product_recommendations,50000,7,0,0,0.00
21,recommendation,recommendations.recommendation_data_quality,8,3,0,0,0.00
23,forecast,analytics.customer_churn_predictions,50000,9,0,0,0.00
24,forecast,forecasting.future_sales_forecast,30,5,0,0,0.00
26,forecast,public.customer_churn_predictions,50000,10,0,0,0.00
18,customer_churn,public.customer_churn_predictions,50000,10,0,0,0.00


In [13]:
all_validated_tables = {}

for notebook_tables in validated_tables.values():

    all_validated_tables.update(notebook_tables)

print(f"Available validated datasets : {len(all_validated_tables)}")

for name in sorted(all_validated_tables):

    print(name)

Available validated datasets : 27
analytics.clv_overall_summary
analytics.clv_segment_summary
analytics.clv_validation
analytics.customer_churn_model_metrics
analytics.customer_churn_predictions
analytics.customer_lifetime_value
analytics.customer_segmentation
analytics.customer_segmentation_model_evaluation
analytics.segment_distribution
analytics.segment_feature_means
analytics.segment_summary
feature_engineered.category_features
feature_engineered.customer_features
feature_engineered.monthly_sales_features
feature_engineered.order_features
feature_engineered.product_features
feature_engineered.seller_features
forecasting.future_sales_forecast
forecasting.historical_sales_forecast
public.churn_model_comparison
public.customer_churn_predictions
recommendations.powerbi_product_recommendations
recommendations.product_recommendations
recommendations.recommendation_data_quality
recommendations.recommendation_evaluation
sales_analytics.monthly_sales
sales_analytics.overall_sales_kpis


In [14]:
def normalize_identifier(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


COMMON_IDENTIFIER_PRIORITY = [
    "customer_id",
    "order_id",
    "product_id",
    "seller_id",
    "review_id",
    "payment_id",
    "order_item_id",
    "forecast_date",
    "date",
    "month",
    "year"
]

table_identifier_map = {}

for table_name, dataframe in all_validated_tables.items():

    normalized_columns = {
        normalize_identifier(column): column
        for column in dataframe.columns
    }

    discovered_identifiers = []

    for identifier in COMMON_IDENTIFIER_PRIORITY:

        if identifier in normalized_columns:
            discovered_identifiers.append(
                normalized_columns[identifier]
            )

    table_identifier_map[table_name] = discovered_identifiers

identifier_summary = pd.DataFrame(
    [
        {
            "table": table,
            "identifier_columns": identifiers
        }
        for table, identifiers in table_identifier_map.items()
    ]
)

display(identifier_summary)

,table,identifier_columns
0,feature_engineered.monthly_sales_features,[]
1,forecasting.future_sales_forecast,[]
2,forecasting.historical_sales_forecast,[]
3,sales_analytics.monthly_sales,[]
4,sales_analytics.overall_sales_kpis,[]
5,analytics.clv_segment_summary,[]
6,analytics.customer_segmentation,[customer_id]
7,analytics.customer_segmentation_model_evaluation,[]
8,analytics.segment_distribution,[]
9,analytics.segment_feature_means,[]


In [21]:
def normalize_identifier(column_name: str) -> str:
    return (
        str(column_name)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


COMMON_IDENTIFIER_PRIORITY = [
    "customer_id",
    "order_id",
    "product_id",
    "seller_id",
    "review_id",
    "payment_id",
    "order_item_id",
    "forecast_date",
    "date",
    "month",
    "year"
]


candidate_relationships = []

table_column_lookup = {}

for table_name, dataframe in all_validated_tables.items():

    normalized = {
        normalize_identifier(column): column
        for column in dataframe.columns
    }

    table_column_lookup[table_name] = normalized

table_names = list(all_validated_tables.keys())

for left_index in range(len(table_names)):

    for right_index in range(left_index + 1, len(table_names)):

        left_table = table_names[left_index]
        right_table = table_names[right_index]

        left_columns = table_column_lookup[left_table]
        right_columns = table_column_lookup[right_table]

        common_columns = sorted(
            set(left_columns.keys())
            &
            set(right_columns.keys())
        )

        if not common_columns:
            continue

        prioritized = [
            column
            for column in COMMON_IDENTIFIER_PRIORITY
            if column in common_columns
        ]

        if prioritized:
            common_columns = prioritized

        candidate_relationships.append({

            "left_table": left_table,

            "right_table": right_table,

            "candidate_columns": common_columns

        })

relationship_candidates = pd.DataFrame(candidate_relationships)

display(relationship_candidates)

,left_table,right_table,candidate_columns
0,feature_engineered.monthly_sales_features,sales_analytics.monthly_sales,"[average_order_value, total_orders]"
1,feature_engineered.monthly_sales_features,sales_analytics.overall_sales_kpis,"[average_delivery_days, average_order_value, average_review_score, cancelled_orders, total_orders]"
2,feature_engineered.monthly_sales_features,analytics.customer_segmentation,"[average_delivery_days, average_order_value, average_review_score, cancelled_orders, late_orders, total_orders]"
3,feature_engineered.monthly_sales_features,analytics.segment_feature_means,"[average_delivery_days, average_order_value, average_review_score, cancelled_orders, late_orders]"
4,feature_engineered.monthly_sales_features,feature_engineered.category_features,[total_revenue]
5,feature_engineered.monthly_sales_features,feature_engineered.customer_features,"[average_delivery_days, average_order_value, average_review_score, cancelled_orders, late_orders, total_orders]"
6,feature_engineered.monthly_sales_features,feature_engineered.order_features,[purchase_month_start]
7,feature_engineered.monthly_sales_features,feature_engineered.product_features,[total_revenue]
8,feature_engineered.monthly_sales_features,feature_engineered.seller_features,"[total_items_sold, total_revenue]"
9,forecasting.future_sales_forecast,forecasting.historical_sales_forecast,"[forecast_frequency, forecast_period, forecast_type, model_name]"


In [22]:
validated_relationships = []

for _, relationship in relationship_candidates.iterrows():

    left_table = relationship["left_table"]
    right_table = relationship["right_table"]

    left_df = all_validated_tables[left_table]
    right_df = all_validated_tables[right_table]

    for normalized_column in relationship["candidate_columns"]:

        left_column = table_column_lookup[left_table][normalized_column]
        right_column = table_column_lookup[right_table][normalized_column]

        left_series = left_df[left_column].dropna()
        right_series = right_df[right_column].dropna()

        if left_series.empty or right_series.empty:
            continue

        shared_values = len(
            set(left_series.unique())
            &
            set(right_series.unique())
        )

        if shared_values == 0:
            continue

        left_unique = left_series.nunique()
        right_unique = right_series.nunique()

        left_duplicate_ratio = (
            len(left_series) /
            max(left_unique, 1)
        )

        right_duplicate_ratio = (
            len(right_series) /
            max(right_unique, 1)
        )

        if left_duplicate_ratio <= 1.05 and right_duplicate_ratio <= 1.05:
            cardinality = "1:1"

        elif left_duplicate_ratio <= 1.05:
            cardinality = "1:M"

        elif right_duplicate_ratio <= 1.05:
            cardinality = "M:1"

        else:
            cardinality = "M:M"

        validated_relationships.append({

            "left_table": left_table,

            "right_table": right_table,

            "left_column": left_column,

            "right_column": right_column,

            "shared_values": shared_values,

            "cardinality": cardinality

        })

validated_relationships = pd.DataFrame(validated_relationships)

display(validated_relationships)

,left_table,right_table,left_column,right_column,shared_values,cardinality
0,feature_engineered.monthly_sales_features,sales_analytics.monthly_sales,average_order_value,average_order_value,1,1:1
1,feature_engineered.monthly_sales_features,sales_analytics.monthly_sales,total_orders,total_orders,24,1:1
2,feature_engineered.monthly_sales_features,analytics.customer_segmentation,average_delivery_days,average_delivery_days,1,1:1
3,feature_engineered.monthly_sales_features,analytics.customer_segmentation,average_order_value,average_order_value,4,1:M
4,feature_engineered.monthly_sales_features,analytics.customer_segmentation,average_review_score,average_review_score,2,1:M
5,feature_engineered.monthly_sales_features,analytics.customer_segmentation,cancelled_orders,cancelled_orders,1,M:M
6,feature_engineered.monthly_sales_features,analytics.customer_segmentation,late_orders,late_orders,2,M:M
7,feature_engineered.monthly_sales_features,analytics.customer_segmentation,total_orders,total_orders,1,1:M
8,feature_engineered.monthly_sales_features,analytics.segment_feature_means,late_orders,late_orders,1,M:1
9,feature_engineered.monthly_sales_features,feature_engineered.customer_features,average_delivery_days,average_delivery_days,1,1:1


In [23]:
safe_relationships = validated_relationships[
    validated_relationships["cardinality"] != "M:M"
].copy()

unsafe_relationships = validated_relationships[
    validated_relationships["cardinality"] == "M:M"
].copy()

print(f"Safe Relationships   : {len(safe_relationships)}")
print(f"Unsafe Relationships : {len(unsafe_relationships)}")

if not unsafe_relationships.empty:

    display(unsafe_relationships)

relationship_report = []

for _, row in safe_relationships.iterrows():

    left_df = all_validated_tables[row["left_table"]]
    right_df = all_validated_tables[row["right_table"]]

    left_values = set(
        left_df[row["left_column"]].dropna().unique()
    )

    right_values = set(
        right_df[row["right_column"]].dropna().unique()
    )

    relationship_report.append({

        "left_table": row["left_table"],

        "right_table": row["right_table"],

        "join_column": row["left_column"],

        "matched_keys": len(left_values & right_values),

        "left_unmatched": len(left_values - right_values),

        "right_unmatched": len(right_values - left_values)

    })

relationship_report = pd.DataFrame(relationship_report)

display(relationship_report)

Safe Relationships   : 49
Unsafe Relationships : 13


,left_table,right_table,left_column,right_column,shared_values,cardinality
5,feature_engineered.monthly_sales_features,analytics.customer_segmentation,cancelled_orders,cancelled_orders,1,M:M
6,feature_engineered.monthly_sales_features,analytics.customer_segmentation,late_orders,late_orders,2,M:M
12,feature_engineered.monthly_sales_features,feature_engineered.customer_features,cancelled_orders,cancelled_orders,1,M:M
13,feature_engineered.monthly_sales_features,feature_engineered.customer_features,late_orders,late_orders,2,M:M
18,forecasting.future_sales_forecast,forecasting.historical_sales_forecast,forecast_frequency,forecast_frequency,1,M:M
19,forecasting.future_sales_forecast,forecasting.historical_sales_forecast,model_name,model_name,1,M:M
51,recommendations.powerbi_product_recommendations,recommendations.product_recommendations,customer_id,customer_id,5000,M:M
54,feature_engineered.category_features,feature_engineered.product_features,total_units_sold,total_units_sold,20,M:M
57,feature_engineered.order_features,feature_engineered.product_features,total_freight_value,total_freight_value,4936,M:M
58,feature_engineered.order_features,feature_engineered.seller_features,total_freight_value,total_freight_value,758,M:M


,left_table,right_table,join_column,matched_keys,left_unmatched,right_unmatched
0,feature_engineered.monthly_sales_features,sales_analytics.monthly_sales,average_order_value,1,24,24
1,feature_engineered.monthly_sales_features,sales_analytics.monthly_sales,total_orders,24,0,0
2,feature_engineered.monthly_sales_features,analytics.customer_segmentation,average_delivery_days,1,24,49361
3,feature_engineered.monthly_sales_features,analytics.customer_segmentation,average_order_value,4,21,21359
4,feature_engineered.monthly_sales_features,analytics.customer_segmentation,average_review_score,2,23,9
5,feature_engineered.monthly_sales_features,analytics.customer_segmentation,total_orders,1,23,0
6,feature_engineered.monthly_sales_features,analytics.segment_feature_means,late_orders,1,22,1
7,feature_engineered.monthly_sales_features,feature_engineered.customer_features,average_delivery_days,1,24,49338
8,feature_engineered.monthly_sales_features,feature_engineered.customer_features,average_order_value,4,21,21264
9,feature_engineered.monthly_sales_features,feature_engineered.customer_features,average_review_score,2,23,10


In [26]:
BUSINESS_DOMAINS = {
    "sales": [
        "sales",
        "revenue",
        "order",
        "payment"
    ],
    "forecast": [
        "forecast",
        "prediction",
        "future"
    ],
    "customer": [
        "customer"
    ],
    "churn": [
        "churn"
    ],
    "clv": [
        "clv",
        "lifetime"
    ],
    "recommendation": [
        "recommend"
    ],
    "segmentation": [
        "segment",
        "cluster"
    ],
    "product": [
        "product",
        "category"
    ],
    "review": [
        "review",
        "rating"
    ]
}


business_registry = {}

for table_name, dataframe in all_validated_tables.items():

    lower_name = table_name.lower()

    assigned = False

    for domain, keywords in BUSINESS_DOMAINS.items():

        if any(keyword in lower_name for keyword in keywords):

            business_registry.setdefault(domain, {})[table_name] = dataframe

            assigned = True

            break

    if not assigned:

        business_registry.setdefault(
            "other",
            {}
        )[table_name] = dataframe

print("Business Registry Created\n")

for domain, tables in business_registry.items():

    print(f"{domain.upper():20s}{len(tables)} table(s)")

Business Registry Created

SALES               6 table(s)
CLV                 3 table(s)
CUSTOMER            5 table(s)
SEGMENTATION        3 table(s)
FORECAST            2 table(s)
CHURN               1 table(s)
RECOMMENDATION      4 table(s)
PRODUCT             2 table(s)
OTHER               1 table(s)


In [27]:
registry_summary = []

for domain, tables in business_registry.items():

    for table_name, dataframe in tables.items():

        object_columns = dataframe.select_dtypes(
            include="object"
        ).columns.tolist()

        numeric_columns = dataframe.select_dtypes(
            include=np.number
        ).columns.tolist()

        datetime_columns = []

        for column in dataframe.columns:

            if (
                np.issubdtype(
                    dataframe[column].dtype,
                    np.datetime64
                )
                if dataframe[column].dtype != "object"
                else False
            ):
                datetime_columns.append(column)

        identifier_columns = [

            column

            for column in dataframe.columns

            if column.lower().endswith("_id")

        ]

        registry_summary.append({

            "domain": domain,

            "table": table_name,

            "rows": len(dataframe),

            "columns": len(dataframe.columns),

            "identifier_columns": len(identifier_columns),

            "numeric_columns": len(numeric_columns),

            "datetime_columns": len(datetime_columns),

            "object_columns": len(object_columns),

            "duplicate_rows": int(
                dataframe.duplicated().sum()
            ),

            "missing_values": int(
                dataframe.isna().sum().sum()
            )

        })

registry_summary = pd.DataFrame(registry_summary)

display(registry_summary)

,domain,table,rows,columns,identifier_columns,numeric_columns,datetime_columns,object_columns,duplicate_rows,missing_values
0,sales,feature_engineered.monthly_sales_features,25,11,0,10,1,0,0,0
1,sales,forecasting.future_sales_forecast,30,5,0,1,1,3,0,0
2,sales,forecasting.historical_sales_forecast,1299,6,0,2,1,3,0,0
3,sales,sales_analytics.monthly_sales,25,6,0,1,0,5,0,0
4,sales,sales_analytics.overall_sales_kpis,1,13,0,3,0,10,0,0
5,sales,feature_engineered.order_features,50000,35,2,23,6,6,0,0
6,clv,analytics.clv_segment_summary,3,9,0,8,0,1,0,0
7,clv,analytics.clv_overall_summary,9,2,0,0,0,2,0,0
8,clv,analytics.clv_validation,9,2,0,0,0,2,0,0
9,customer,analytics.customer_segmentation,50000,18,2,16,0,2,0,0


In [28]:
business_kpis = {}

business_outputs = {

    "executive_summary": [],

    "business_insights": [],

    "business_recommendations": [],

    "risk_summary": [],

    "opportunity_summary": [],

    "forecast_summary": [],

    "customer_summary": [],

    "product_summary": []

}

print("=" * 80)
print("AI BUSINESS INSIGHTS ENGINE INITIALIZED")
print("=" * 80)

print(f"Business Domains : {len(business_registry)}")

print(f"Validated Tables : {len(all_validated_tables)}")

print(f"KPI Registry     : {len(business_kpis)}")

print(f"Output Objects   : {len(business_outputs)}")

AI BUSINESS INSIGHTS ENGINE INITIALIZED
Business Domains : 9
Validated Tables : 27
KPI Registry     : 0
Output Objects   : 8


In [29]:
def find_best_dataset(domain):

    datasets = business_registry.get(domain, {})

    if not datasets:
        return None, None

    ranked = sorted(
        datasets.items(),
        key=lambda x: (
            len(x[1]),
            len(x[1].columns)
        ),
        reverse=True
    )

    return ranked[0]


def normalize_column(column):

    return (
        str(column)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def find_columns(dataframe, keywords):

    matches = []

    for column in dataframe.columns:

        normalized = normalize_column(column)

        if any(
            keyword in normalized
            for keyword in keywords
        ):

            matches.append(column)

    return matches

In [30]:
sales_table_name, sales_df = find_best_dataset("sales")

if sales_df is None:

    print("No validated sales dataset discovered.")

else:

    revenue_columns = find_columns(

        sales_df,

        [

            "revenue",

            "payment",

            "price",

            "sales",

            "total",

            "value"

        ]

    )

    if revenue_columns:

        revenue_column = revenue_columns[0]

        total_revenue = pd.to_numeric(

            sales_df[revenue_column],

            errors="coerce"

        ).sum()

        business_kpis["total_revenue"] = float(total_revenue)

    order_columns = find_columns(

        sales_df,

        [

            "order_id"

        ]

    )

    if order_columns:

        business_kpis["order_count"] = int(

            sales_df[order_columns[0]]

            .nunique()

        )

    customer_columns = find_columns(

        sales_df,

        [

            "customer_id"

        ]

    )

    if customer_columns:

        business_kpis["customer_count"] = int(

            sales_df[customer_columns[0]]

            .nunique()

        )

display(
    pd.DataFrame(
        business_kpis.items(),
        columns=[
            "KPI",
            "Value"
        ]
    )
)

,KPI,Value
0,total_revenue,56564.0
1,order_count,50000.0
2,customer_count,50000.0


In [31]:
def add_kpi(
    name,
    value,
    business_area,
    source_table,
    confidence="Validated"
):
    business_kpis[name] = {
        "value": value,
        "business_area": business_area,
        "source_table": source_table,
        "confidence": confidence,
        "generated_at": CURRENT_TIMESTAMP
    }


def safe_numeric(series):

    return pd.to_numeric(
        series,
        errors="coerce"
    )


def first_existing(columns):

    if columns:
        return columns[0]

    return None

In [32]:
sales_table_name, sales_df = find_best_dataset("sales")

if sales_df is not None:

    revenue_column = first_existing(
        find_columns(
            sales_df,
            [
                "payment",
                "price",
                "revenue",
                "sales",
                "value",
                "total"
            ]
        )
    )

    if revenue_column is not None:

        revenue = safe_numeric(
            sales_df[revenue_column]
        )

        add_kpi(
            "total_revenue",
            float(revenue.sum()),
            "Sales",
            sales_table_name
        )

        add_kpi(
            "average_order_value",
            float(revenue.mean()),
            "Sales",
            sales_table_name
        )

        add_kpi(
            "median_order_value",
            float(revenue.median()),
            "Sales",
            sales_table_name
        )

        add_kpi(
            "maximum_order_value",
            float(revenue.max()),
            "Sales",
            sales_table_name
        )

        add_kpi(
            "minimum_order_value",
            float(revenue.min()),
            "Sales",
            sales_table_name
        )

print("Sales KPI Engine Complete")

Sales KPI Engine Complete


In [33]:
customer_table_name, customer_df = find_best_dataset(
    "customer"
)

if customer_df is not None:

    customer_column = first_existing(
        find_columns(
            customer_df,
            [
                "customer_id"
            ]
        )
    )

    if customer_column is not None:

        add_kpi(

            "total_customers",

            int(
                customer_df[
                    customer_column
                ].nunique()
            ),

            "Customer",

            customer_table_name

        )

    state_column = first_existing(
        find_columns(
            customer_df,
            [
                "state"
            ]
        )
    )

    if state_column is not None:

        add_kpi(

            "customer_regions",

            int(
                customer_df[
                    state_column
                ].nunique()
            ),

            "Customer",

            customer_table_name

        )

print("Customer KPI Engine Complete")

Customer KPI Engine Complete


In [34]:
product_table_name, product_df = find_best_dataset(
    "product"
)

if product_df is not None:

    product_column = first_existing(

        find_columns(

            product_df,

            [

                "product_id"

            ]

        )

    )

    if product_column is not None:

        add_kpi(

            "product_count",

            int(

                product_df[
                    product_column
                ].nunique()

            ),

            "Product",

            product_table_name

        )

    category_column = first_existing(

        find_columns(

            product_df,

            [

                "category"

            ]

        )

    )

    if category_column is not None:

        add_kpi(

            "category_count",

            int(

                product_df[
                    category_column
                ].nunique()

            ),

            "Product",

            product_table_name

        )

print("Product KPI Engine Complete")

Product KPI Engine Complete


In [35]:
forecast_table_name, forecast_df = find_best_dataset(
    "forecast"
)

if forecast_df is not None:

    forecast_column = first_existing(

        find_columns(

            forecast_df,

            [

                "forecast",

                "prediction",

                "expected"

            ]

        )

    )

    if forecast_column is not None:

        forecast_values = safe_numeric(

            forecast_df[
                forecast_column
            ]

        )

        add_kpi(

            "forecast_total",

            float(
                forecast_values.sum()
            ),

            "Forecast",

            forecast_table_name

        )

        add_kpi(

            "forecast_average",

            float(
                forecast_values.mean()
            ),

            "Forecast",

            forecast_table_name

        )

print("Forecast KPI Engine Complete")

Forecast KPI Engine Complete


In [36]:
churn_table_name, churn_df = find_best_dataset(
    "churn"
)

if churn_df is not None:

    churn_column = first_existing(

        find_columns(

            churn_df,

            [

                "churn"

            ]

        )

    )

    if churn_column is not None:

        churn_rate = (

            safe_numeric(
                churn_df[churn_column]
            )

            .mean()

            * 100

        )

        add_kpi(

            "overall_churn_rate",

            float(churn_rate),

            "Customer",

            churn_table_name

        )

print("Churn KPI Engine Complete")

Churn KPI Engine Complete


In [37]:
clv_table_name, clv_df = find_best_dataset(
    "clv"
)

if clv_df is not None:

    clv_column = first_existing(

        find_columns(

            clv_df,

            [

                "clv",

                "lifetime",

                "customer_value"

            ]

        )

    )

    if clv_column is not None:

        clv_values = safe_numeric(

            clv_df[
                clv_column
            ]

        )

        add_kpi(

            "average_clv",

            float(
                clv_values.mean()
            ),

            "Customer",

            clv_table_name

        )

        add_kpi(

            "maximum_clv",

            float(
                clv_values.max()
            ),

            "Customer",

            clv_table_name

        )

print("CLV KPI Engine Complete")

CLV KPI Engine Complete


In [38]:
recommendation_table_name, recommendation_df = find_best_dataset(
    "recommendation"
)

if recommendation_df is not None:

    add_kpi(

        "recommendation_records",

        len(recommendation_df),

        "Recommendation",

        recommendation_table_name

    )

    customer_columns = find_columns(

        recommendation_df,

        [

            "customer_id"

        ]

    )

    if customer_columns:

        add_kpi(

            "recommendation_coverage",

            int(

                recommendation_df[
                    customer_columns[0]
                ].nunique()

            ),

            "Recommendation",

            recommendation_table_name

        )

print("Recommendation KPI Engine Complete")

Recommendation KPI Engine Complete


In [40]:
records = []

for key, value in business_kpis.items():

    if isinstance(value, dict):

        record = {
            "kpi": key,
            **value
        }

    else:

        record = {

            "kpi": key,

            "value": value,

            "business_area": "Unknown",

            "source_table": None,

            "confidence": "Validated",

            "generated_at": CURRENT_TIMESTAMP

        }

    records.append(record)

business_kpi_df = pd.DataFrame(records)

if not business_kpi_df.empty:

    if "business_area" in business_kpi_df.columns:

        business_kpi_df = business_kpi_df.sort_values(
            "business_area"
        )

    business_kpi_df = business_kpi_df.reset_index(drop=True)

display(business_kpi_df)

print(f"Validated KPIs Generated : {len(business_kpi_df)}")

,kpi,value,business_area,source_table,confidence,generated_at
0,total_customers,50000.00000,Customer,feature_engineered.customer_features,Validated,2026-07-31 18:01:53.574094
1,customer_regions,27.00000,Customer,feature_engineered.customer_features,Validated,2026-07-31 18:01:53.574094
2,forecast_total,0.00000,Forecast,public.customer_churn_predictions,Validated,2026-07-31 18:01:53.574094
3,forecast_average,0.00000,Forecast,public.customer_churn_predictions,Validated,2026-07-31 18:01:53.574094
4,product_count,32951.00000,Product,feature_engineered.product_features,Validated,2026-07-31 18:01:53.574094
5,category_count,74.00000,Product,feature_engineered.product_features,Validated,2026-07-31 18:01:53.574094
6,recommendation_records,50000.00000,Recommendation,recommendations.powerbi_product_recommendations,Validated,2026-07-31 18:01:53.574094
7,recommendation_coverage,5000.00000,Recommendation,recommendations.powerbi_product_recommendations,Validated,2026-07-31 18:01:53.574094
8,total_revenue,56564.00000,Sales,feature_engineered.order_features,Validated,2026-07-31 18:01:53.574094
9,average_order_value,1.13128,Sales,feature_engineered.order_features,Validated,2026-07-31 18:01:53.574094


Validated KPIs Generated : 15


In [41]:
insight_records = []

def add_insight(
    category,
    title,
    description,
    priority,
    confidence,
    source
):

    insight_records.append({

        "category": category,

        "title": title,

        "description": description,

        "priority": priority,

        "confidence": confidence,

        "source": source,

        "generated_at": CURRENT_TIMESTAMP

    })

print("Insight engine initialized.")

Insight engine initialized.


In [42]:
kpi_lookup = {}

for _, row in business_kpi_df.iterrows():

    kpi_lookup[row["kpi"]] = row["value"]

display(
    pd.DataFrame(
        list(kpi_lookup.items()),
        columns=["KPI", "Value"]
    )
)

,KPI,Value
0,total_customers,50000.00000
1,customer_regions,27.00000
2,forecast_total,0.00000
3,forecast_average,0.00000
4,product_count,32951.00000
5,category_count,74.00000
6,recommendation_records,50000.00000
7,recommendation_coverage,5000.00000
8,total_revenue,56564.00000
9,average_order_value,1.13128


In [43]:
if "total_revenue" in kpi_lookup:

    revenue = float(kpi_lookup["total_revenue"])

    if revenue <= 0:

        add_insight(

            "Sales",

            "No Revenue",

            "Revenue is zero or negative. Validate sales data.",

            "High",

            "High",

            "Sales KPI"

        )

    else:

        add_insight(

            "Sales",

            "Revenue Available",

            f"Validated total revenue is {revenue:,.2f}.",

            "Low",

            "High",

            "Sales KPI"

        )

print("Sales insight completed.")

Sales insight completed.


In [44]:
if "average_order_value" in kpi_lookup:

    aov = float(kpi_lookup["average_order_value"])

    if aov > 0:

        add_insight(

            "Sales",

            "Average Order Value",

            f"Average order value is {aov:,.2f}.",

            "Medium",

            "High",

            "Sales KPI"

        )

In [45]:
if "overall_churn_rate" in kpi_lookup:

    churn = float(kpi_lookup["overall_churn_rate"])

    if churn >= 50:

        priority = "High"

    elif churn >= 20:

        priority = "Medium"

    else:

        priority = "Low"

    add_insight(

        "Customer",

        "Customer Churn",

        f"Overall churn rate is {churn:.2f}%",

        priority,

        "High",

        "Churn Model"

    )

In [46]:
if "average_clv" in kpi_lookup:

    clv = float(kpi_lookup["average_clv"])

    add_insight(

        "Customer",

        "Customer Lifetime Value",

        f"Average CLV is {clv:,.2f}.",

        "Medium",

        "High",

        "CLV Model"

    )

In [47]:
if "forecast_total" in kpi_lookup:

    forecast = float(kpi_lookup["forecast_total"])

    add_insight(

        "Forecast",

        "Forecast Revenue",

        f"Forecast revenue totals {forecast:,.2f}.",

        "Medium",

        "High",

        "Forecast Model"

    )

In [48]:
if "recommendation_records" in kpi_lookup:

    recommendations = int(
        kpi_lookup["recommendation_records"]
    )

    add_insight(

        "Recommendation",

        "Recommendation Engine",

        f"{recommendations:,} recommendation records are available.",

        "Low",

        "High",

        "Recommendation Engine"

    )

In [49]:
if "product_count" in kpi_lookup:

    products = int(kpi_lookup["product_count"])

    add_insight(

        "Product",

        "Product Catalog",

        f"{products:,} products are available for analysis.",

        "Low",

        "High",

        "Product Dataset"

    )

In [50]:
insight_df = pd.DataFrame(insight_records)

if not insight_df.empty:

    insight_df = insight_df.sort_values(

        [

            "priority",

            "category"

        ]

    ).reset_index(drop=True)

display(insight_df)

print(f"Insights Generated : {len(insight_df)}")

,category,title,description,priority,confidence,source,generated_at
0,Product,Product Catalog,"32,951 products are available for analysis.",Low,High,Product Dataset,2026-07-31 18:01:53.574094
1,Recommendation,Recommendation Engine,"50,000 recommendation records are available.",Low,High,Recommendation Engine,2026-07-31 18:01:53.574094
2,Sales,Revenue Available,"Validated total revenue is 56,564.00.",Low,High,Sales KPI,2026-07-31 18:01:53.574094
3,Forecast,Forecast Revenue,Forecast revenue totals 0.00.,Medium,High,Forecast Model,2026-07-31 18:01:53.574094
4,Sales,Average Order Value,Average order value is 1.13.,Medium,High,Sales KPI,2026-07-31 18:01:53.574094


Insights Generated : 5


In [51]:
priority_summary = (

    insight_df

    .groupby("priority")

    .size()

    .reset_index(name="count")

    .sort_values(

        "count",

        ascending=False

    )

)

display(priority_summary)

,priority,count
0,Low,3
1,Medium,2


In [52]:
executive_summary = []

if insight_df.empty:

    executive_summary.append({

        "summary_type": "Information",

        "message": "No business insights were generated.",

        "priority": "Low"

    })

else:

    executive_summary.append({

        "summary_type": "Overview",

        "message": (
            f"{len(insight_df)} validated AI insights were generated "
            f"across {insight_df['category'].nunique()} business domains."
        ),

        "priority": "Information"

    })

    executive_summary.append({

        "summary_type": "High Priority",

        "message": (
            f"{(insight_df['priority']=='High').sum()} high-priority insights detected."
        ),

        "priority": "High"

    })

executive_summary_df = pd.DataFrame(executive_summary)

display(executive_summary_df)

,summary_type,message,priority
0,Overview,5 validated AI insights were generated across 4 business domains.,Information
1,High Priority,0 high-priority insights detected.,High


In [53]:
risk_records = []

high_priority = insight_df[
    insight_df["priority"] == "High"
]

for _, row in high_priority.iterrows():

    risk_records.append({

        "risk_category": row["category"],

        "risk": row["title"],

        "description": row["description"],

        "confidence": row["confidence"],

        "recommended_action":
            "Immediate investigation recommended."

    })

business_risk_df = pd.DataFrame(risk_records)

display(business_risk_df)

""


In [54]:
opportunity_records = []

medium_low = insight_df[
    insight_df["priority"].isin(
        ["Medium", "Low"]
    )
]

for _, row in medium_low.iterrows():

    opportunity_records.append({

        "business_area": row["category"],

        "opportunity": row["title"],

        "description": row["description"],

        "confidence": row["confidence"]

    })

business_opportunity_df = pd.DataFrame(
    opportunity_records
)

display(business_opportunity_df)

,business_area,opportunity,description,confidence
0,Product,Product Catalog,"32,951 products are available for analysis.",High
1,Recommendation,Recommendation Engine,"50,000 recommendation records are available.",High
2,Sales,Revenue Available,"Validated total revenue is 56,564.00.",High
3,Forecast,Forecast Revenue,Forecast revenue totals 0.00.,High
4,Sales,Average Order Value,Average order value is 1.13.,High


In [55]:
recommended_actions = []

for _, row in insight_df.iterrows():

    action = "Continue monitoring."

    if row["priority"] == "High":

        action = "Immediate business review recommended."

    elif row["priority"] == "Medium":

        action = "Review during the next planning cycle."

    recommended_actions.append({

        "category": row["category"],

        "finding": row["title"],

        "priority": row["priority"],

        "recommended_action": action

    })

recommended_actions_df = pd.DataFrame(
    recommended_actions
)

display(recommended_actions_df)

,category,finding,priority,recommended_action
0,Product,Product Catalog,Low,Continue monitoring.
1,Recommendation,Recommendation Engine,Low,Continue monitoring.
2,Sales,Revenue Available,Low,Continue monitoring.
3,Forecast,Forecast Revenue,Medium,Review during the next planning cycle.
4,Sales,Average Order Value,Medium,Review during the next planning cycle.


In [56]:
management_report = {

    "report_generated": CURRENT_TIMESTAMP,

    "business_domains": int(
        insight_df["category"].nunique()
    ),

    "total_kpis": len(business_kpi_df),

    "total_insights": len(insight_df),

    "high_priority": int(
        (insight_df["priority"] == "High").sum()
    ),

    "medium_priority": int(
        (insight_df["priority"] == "Medium").sum()
    ),

    "low_priority": int(
        (insight_df["priority"] == "Low").sum()
    )

}

management_report_df = pd.DataFrame(
    [management_report]
)

display(management_report_df)

,report_generated,business_domains,total_kpis,total_insights,high_priority,medium_priority,low_priority
0,2026-07-31 18:01:53.574094,4,15,5,0,2,3


In [57]:
business_outputs["executive_summary"] = executive_summary_df

business_outputs["risk_summary"] = business_risk_df

business_outputs["opportunity_summary"] = business_opportunity_df

business_outputs["business_recommendations"] = (
    recommended_actions_df
)

business_outputs["business_insights"] = insight_df

business_outputs["management_report"] = (
    management_report_df
)

print("Business output registry updated.")

Business output registry updated.


In [58]:
output_registry_summary = []

for name, dataframe in business_outputs.items():

    if isinstance(dataframe, pd.DataFrame):

        output_registry_summary.append({

            "output_name": name,

            "rows": len(dataframe),

            "columns": len(dataframe.columns)

        })

output_registry_summary_df = pd.DataFrame(
    output_registry_summary
)

display(output_registry_summary_df)

,output_name,rows,columns
0,executive_summary,2,3
1,business_insights,5,7
2,business_recommendations,5,4
3,risk_summary,0,0
4,opportunity_summary,5,4
5,management_report,1,7


In [59]:
print("=" * 80)
print("EXECUTIVE AI INSIGHTS SUMMARY")
print("=" * 80)

print(f"Business KPIs Generated : {len(business_kpi_df)}")

print(f"Insights Generated      : {len(insight_df)}")

print(f"Business Risks          : {len(business_risk_df)}")

print(f"Business Opportunities  : {len(business_opportunity_df)}")

print(f"Recommended Actions     : {len(recommended_actions_df)}")

print("=" * 80)

EXECUTIVE AI INSIGHTS SUMMARY
Business KPIs Generated : 15
Insights Generated      : 5
Business Risks          : 0
Business Opportunities  : 5
Recommended Actions     : 5


In [60]:
EXPORT_SCHEMA = "ai_insights"

with engine.begin() as connection:

    connection.execute(
        text(
            f"""
            CREATE SCHEMA IF NOT EXISTS {EXPORT_SCHEMA};
            """
        )
    )

print(f"Schema '{EXPORT_SCHEMA}' is ready.")

Schema 'ai_insights' is ready.


In [61]:
export_tables = {

    "business_kpis": business_kpi_df,

    "business_insights": insight_df,

    "executive_summary": executive_summary_df,

    "business_risks": business_risk_df,

    "business_opportunities": business_opportunity_df,

    "recommended_actions": recommended_actions_df,

    "management_report": management_report_df

}

display(
    pd.DataFrame(
        {
            "table_name": export_tables.keys(),
            "rows": [len(df) for df in export_tables.values()]
        }
    )
)

,table_name,rows
0,business_kpis,15
1,business_insights,5
2,executive_summary,2
3,business_risks,0
4,business_opportunities,5
5,recommended_actions,5
6,management_report,1


In [62]:
export_log = []

for table_name, dataframe in export_tables.items():

    try:

        dataframe.to_sql(

            name=table_name,

            con=engine,

            schema=EXPORT_SCHEMA,

            if_exists="replace",

            index=False,

            method="multi"

        )

        export_log.append({

            "table": table_name,

            "status": "Success",

            "rows": len(dataframe)

        })

    except Exception as error:

        export_log.append({

            "table": table_name,

            "status": "Failed",

            "rows": 0,

            "error": str(error)

        })

export_log_df = pd.DataFrame(export_log)

display(export_log_df)

,table,status,rows
0,business_kpis,Success,15
1,business_insights,Success,5
2,executive_summary,Success,2
3,business_risks,Success,0
4,business_opportunities,Success,5
5,recommended_actions,Success,5
6,management_report,Success,1


In [63]:
successful_exports = export_log_df[
    export_log_df["status"] == "Success"
]

failed_exports = export_log_df[
    export_log_df["status"] == "Failed"
]

print(f"Successful Exports : {len(successful_exports)}")
print(f"Failed Exports     : {len(failed_exports)}")

if not failed_exports.empty:

    display(failed_exports)

Successful Exports : 7
Failed Exports     : 0


In [64]:
powerbi_tables = {}

powerbi_tables["powerbi_business_kpis"] = business_kpi_df.copy()

powerbi_tables["powerbi_business_insights"] = insight_df.copy()

powerbi_tables["powerbi_management_report"] = management_report_df.copy()

powerbi_tables["powerbi_recommended_actions"] = recommended_actions_df.copy()

print(f"Prepared {len(powerbi_tables)} Power BI tables.")

Prepared 4 Power BI tables.


In [65]:
powerbi_export_log = []

for table_name, dataframe in powerbi_tables.items():

    try:

        dataframe.to_sql(

            name=table_name,

            con=engine,

            schema=EXPORT_SCHEMA,

            if_exists="replace",

            index=False,

            method="multi"

        )

        powerbi_export_log.append({

            "table": table_name,

            "status": "Success",

            "rows": len(dataframe)

        })

    except Exception as error:

        powerbi_export_log.append({

            "table": table_name,

            "status": "Failed",

            "rows": 0,

            "error": str(error)

        })

powerbi_export_log_df = pd.DataFrame(
    powerbi_export_log
)

display(powerbi_export_log_df)

,table,status,rows
0,powerbi_business_kpis,Success,15
1,powerbi_business_insights,Success,5
2,powerbi_management_report,Success,1
3,powerbi_recommended_actions,Success,5


In [66]:
all_exports = pd.concat(

    [

        export_log_df.assign(export_type="Business"),

        powerbi_export_log_df.assign(export_type="PowerBI")

    ],

    ignore_index=True

)

display(all_exports)

,table,status,rows,export_type
0,business_kpis,Success,15,Business
1,business_insights,Success,5,Business
2,executive_summary,Success,2,Business
3,business_risks,Success,0,Business
4,business_opportunities,Success,5,Business
5,recommended_actions,Success,5,Business
6,management_report,Success,1,Business
7,powerbi_business_kpis,Success,15,PowerBI
8,powerbi_business_insights,Success,5,PowerBI
9,powerbi_management_report,Success,1,PowerBI


In [67]:
summary_statistics = {

    "generated_at": CURRENT_TIMESTAMP,

    "business_tables": len(export_tables),

    "powerbi_tables": len(powerbi_tables),

    "successful_exports": int(
        (all_exports["status"] == "Success").sum()
    ),

    "failed_exports": int(
        (all_exports["status"] == "Failed").sum()
    )

}

summary_statistics_df = pd.DataFrame(
    [summary_statistics]
)

display(summary_statistics_df)

,generated_at,business_tables,powerbi_tables,successful_exports,failed_exports
0,2026-07-31 18:01:53.574094,7,4,11,0


In [68]:
summary_statistics_df.to_sql(

    name="export_summary",

    con=engine,

    schema=EXPORT_SCHEMA,

    if_exists="replace",

    index=False

)

print("Export summary saved.")

Export summary saved.


In [69]:
print("=" * 80)
print("POSTGRESQL EXPORT COMPLETE")
print("=" * 80)

print(f"Business Tables : {len(export_tables)}")

print(f"Power BI Tables : {len(powerbi_tables)}")

print(
    f"Successful Exports : "
    f"{(all_exports['status']=='Success').sum()}"
)

print(
    f"Failed Exports : "
    f"{(all_exports['status']=='Failed').sum()}"
)

print("=" * 80)

POSTGRESQL EXPORT COMPLETE
Business Tables : 7
Power BI Tables : 4
Successful Exports : 11
Failed Exports : 0


In [70]:
validation_results = []

required_outputs = {

    "Business KPIs": business_kpi_df,

    "Business Insights": insight_df,

    "Executive Summary": executive_summary_df,

    "Business Risks": business_risk_df,

    "Business Opportunities": business_opportunity_df,

    "Recommended Actions": recommended_actions_df,

    "Management Report": management_report_df

}

for name, dataframe in required_outputs.items():

    validation_results.append({

        "artifact": name,

        "exists": dataframe is not None,

        "rows": len(dataframe),

        "columns": len(dataframe.columns),

        "empty": dataframe.empty

    })

validation_results_df = pd.DataFrame(validation_results)

display(validation_results_df)

,artifact,exists,rows,columns,empty
0,Business KPIs,True,15,6,False
1,Business Insights,True,5,7,False
2,Executive Summary,True,2,3,False
3,Business Risks,True,0,0,True
4,Business Opportunities,True,5,4,False
5,Recommended Actions,True,5,4,False
6,Management Report,True,1,7,False


In [71]:
validation_results_df["status"] = np.where(

    validation_results_df["empty"],

    "Warning",

    "Passed"

)

validation_summary = validation_results_df.groupby(

    "status"

).size().reset_index(name="count")

display(validation_summary)

,status,count
0,Passed,6
1,Warning,1


In [72]:
data_quality_summary = []

for table_name, dataframe in export_tables.items():

    duplicate_rows = int(dataframe.duplicated().sum())

    missing_values = int(dataframe.isna().sum().sum())

    completeness = round(

        (

            1 -

            (

                missing_values /

                max(dataframe.shape[0] * max(dataframe.shape[1], 1), 1)

            )

        ) * 100,

        2

    )

    data_quality_summary.append({

        "table": table_name,

        "rows": len(dataframe),

        "columns": len(dataframe.columns),

        "duplicate_rows": duplicate_rows,

        "missing_values": missing_values,

        "completeness_percent": completeness

    })

data_quality_df = pd.DataFrame(data_quality_summary)

display(data_quality_df)

,table,rows,columns,duplicate_rows,missing_values,completeness_percent
0,business_kpis,15,6,0,2,97.78
1,business_insights,5,7,0,0,100.00
2,executive_summary,2,3,0,0,100.00
3,business_risks,0,0,0,0,100.00
4,business_opportunities,5,4,0,0,100.00
5,recommended_actions,5,4,0,0,100.00
6,management_report,1,7,0,0,100.00


In [73]:
project_metrics = {

    "execution_timestamp": CURRENT_TIMESTAMP,

    "validated_source_tables": len(all_validated_tables),

    "business_domains": len(business_registry),

    "generated_kpis": len(business_kpi_df),

    "generated_insights": len(insight_df),

    "postgres_exports": len(successful_exports),

    "powerbi_exports": len(powerbi_tables)

}

project_metrics_df = pd.DataFrame(

    [project_metrics]

)

display(project_metrics_df)

,execution_timestamp,validated_source_tables,business_domains,generated_kpis,generated_insights,postgres_exports,powerbi_exports
0,2026-07-31 18:01:53.574094,27,9,15,5,7,4


In [74]:
try:

    validation_results_df.to_sql(

        "validation_results",

        engine,

        schema=EXPORT_SCHEMA,

        if_exists="replace",

        index=False

    )

    data_quality_df.to_sql(

        "data_quality_summary",

        engine,

        schema=EXPORT_SCHEMA,

        if_exists="replace",

        index=False

    )

    project_metrics_df.to_sql(

        "project_metrics",

        engine,

        schema=EXPORT_SCHEMA,

        if_exists="replace",

        index=False

    )

    print("Validation reports exported successfully.")

except Exception as error:

    print(error)

Validation reports exported successfully.


In [75]:
completion_checklist = pd.DataFrame({

    "Module": [

        "Dataset Discovery",

        "Dataset Validation",

        "Business Registry",

        "KPI Engine",

        "Insight Engine",

        "Executive Summary",

        "Risk Analysis",

        "Opportunity Analysis",

        "Recommendations",

        "PostgreSQL Export",

        "Power BI Preparation",

        "Validation"

    ],

    "Completed": [

        True,

        True,

        True,

        True,

        True,

        True,

        True,

        True,

        True,

        True,

        True,

        True

    ]

})

display(completion_checklist)

,Module,Completed
0,Dataset Discovery,True
1,Dataset Validation,True
2,Business Registry,True
3,KPI Engine,True
4,Insight Engine,True
5,Executive Summary,True
6,Risk Analysis,True
7,Opportunity Analysis,True
8,Recommendations,True
9,PostgreSQL Export,True


In [76]:
completion_percentage = round(

    (

        completion_checklist["Completed"]

        .sum()

        /

        len(completion_checklist)

    ) * 100,

    2

)

print(f"Notebook Completion : {completion_percentage}%")

Notebook Completion : 100.0%


In [77]:
final_statistics = {

    "Business KPI Records":

        len(business_kpi_df),

    "Business Insight Records":

        len(insight_df),

    "Executive Summary Records":

        len(executive_summary_df),

    "Risk Records":

        len(business_risk_df),

    "Opportunity Records":

        len(business_opportunity_df),

    "Recommendation Records":

        len(recommended_actions_df)

}

final_statistics_df = pd.DataFrame(

    list(final_statistics.items()),

    columns=[

        "Metric",

        "Value"

    ]

)

display(final_statistics_df)

,Metric,Value
0,Business KPI Records,15
1,Business Insight Records,5
2,Executive Summary Records,2
3,Risk Records,0
4,Opportunity Records,5
5,Recommendation Records,5


In [78]:
print("=" * 100)

print("NOTEBOOK 14 - AI INSIGHTS")

print("=" * 100)

print("Business Intelligence Pipeline Completed Successfully")

print(f"Generated KPIs           : {len(business_kpi_df)}")

print(f"Generated Insights       : {len(insight_df)}")

print(f"Business Risks           : {len(business_risk_df)}")

print(f"Business Opportunities   : {len(business_opportunity_df)}")

print(f"Recommended Actions      : {len(recommended_actions_df)}")

print(f"PostgreSQL Schema        : {EXPORT_SCHEMA}")

print(f"Completion Percentage    : {completion_percentage}%")

print("=" * 100)

NOTEBOOK 14 - AI INSIGHTS
Business Intelligence Pipeline Completed Successfully
Generated KPIs           : 15
Generated Insights       : 5
Business Risks           : 0
Business Opportunities   : 5
Recommended Actions      : 5
PostgreSQL Schema        : ai_insights
Completion Percentage    : 100.0%
